# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset via Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Keywords:", getattr(metadata, 'keywords', []))
print("RecordSet IDs:", getattr(metadata, 'recordSet', []))


## 2. Data Overview
Review available record sets, fields, and their IDs.

- Each Record Set and Field is referenced strictly by their `@id`.
- The following code lists available record sets and fields including their IDs from the dataset.

In [ ]:
# Explore available record sets
record_sets = getattr(metadata, 'recordSet', [])

# Print info for each RecordSet and list their fields
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}\n")
    print(f"  Name: {rs.get('name','N/A')}")
    fields = rs.get('field', [])
    print("  Fields/Columns:")
    for f in fields:
        print(f"    Field @id: {f.get('@id')} - name: {f.get('name', 'N/A')} - dataType: {f.get('dataType', 'N/A')}")
    print('-'*30)

# Example: For loading records, choose the main tabular RecordSet @id
# (Below we extract all RecordSet @id values)
record_set_ids = [rs['@id'] for rs in record_sets]


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

- All loading uses the `@id` reference.
- Example below loads all available record sets into individual pandas DataFrames.

In [ ]:
# Extract data from each RecordSet using its @id
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns of the main record set
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id and main_rs_id in dataframes:
    print(f"Columns in the main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    print(dataframes[main_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- All entities referenced by `@id`.
- Please adjust the numeric field and grouping field IDs (below are examples).

In [ ]:
# Select a numeric field for analysis: GAD-7 score
# Example @id (update based on metadata output):
numeric_field_id = None
group_field_id = None

# Find a numeric field (e.g., GAD-7 score column)
# Loop through record set to find a suitable numeric field
for rs in record_sets:
    fields = rs.get('field', [])
    for f in fields:
        if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
            # Example matching GAD-7, PHQ-9, PSQ field
            if 'gad' in f.get('name','').lower() or 'score' in f.get('name','').lower():
                numeric_field_id = f['@id']
            if 'village' in f.get('name','').lower() or 'education' in f.get('name','').lower():
                group_field_id = f['@id']
if not numeric_field_id:
    # Fallback: use any numeric field
    for rs in record_sets:
        for f in rs.get('field', []):
            if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
                numeric_field_id = f['@id']
                break
        if numeric_field_id:
            break

# Use the first available record set
record_set_id = main_rs_id
df = dataframes[record_set_id]
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if available
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field if available
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot if group field exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization cannot be produced: numeric field missing.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides AI-ready data with rich demographics and psychological indicator scores.
- Sample EDA and visualizations highlight typical analytic workflows using Croissant schemas and `mlcroissant` for efficient, standards-based data loading and referencing.
- All entities (record sets, fields) are referenced using their `@id`, supporting interoperability and reproducibility.
- You can extend this notebook for deeper modeling, integration, or cross-dataset comparison via Croissant-compliant workflows.

For more information, see [mlcroissant documentation](https://github.com/mlcommons/croissant).